In [10]:
"""
Generating a Corporate Credit Rating Dataset for Classification
The dataset is composed by Financials + Credit Rating

Sources of data:
https://public.opendatasoft.com/
https://financialmodelingprep.com/
"""

# Import Libraries
import pandas as pd
import re
import json
from urllib.request import urlopen

# Import Data
df_is_stocks = pd.read_csv('us_stocks.csv')
df_ratings = pd.read_csv('ratings-history.csv')

# Drop NA
df_ratings = df_ratings.dropna()

# Drop Duplicates
df_ratings = df_ratings.drop_duplicates()

# Standarize ratings
def standarizeRating(name):
    
    name = re.sub(r'Aaa','AAA', name)
    name = re.sub(r'Aa1', 'AA+', name)
    name = re.sub(r'Aa2', 'AA', name)
    name = re.sub(r'Aa3	', 'AA−', name)
    name = re.sub(r'A1', 'A+', name)    
    name = re.sub(r'A2','A', name)   
    name = re.sub(r'A3', 'A-', name)
    name = re.sub(r'Baa1','BBB+', name)
    name = re.sub(r'Baa2','BBB', name)
    name = re.sub(r'Baa3', 'BBB-', name)
    name = re.sub(r'Ba1','BB+', name)
    name = re.sub(r'Ba2', 'BB', name)
    name = re.sub(r'Ba3', 'BB−', name)
    name = re.sub(r'B1','B+', name)    
    name = re.sub(r'B2','B', name)   
    name = re.sub(r'B3','B−', name)
    name = re.sub(r'Caa1','CCC+', name)
    name = re.sub(r'Caa2','CCC', name)
    name = re.sub(r'Caa3', 'CCC−', name)
    return name

df_ratings


,Rating Agency Name,Name,Rating Action Date,Rating
0,Egan-Jones Ratings Company,TELECOM ITALIA S.P.A,8/26/2014,BB-
1,Egan-Jones Ratings Company,WHIRLPOOL CORPORATION,11/27/2015,A-
2,Egan-Jones Ratings Company,UNITED TECHNOLOGIES,7/28/2014,A
3,Fitch Ratings,TEPPCO Partners L.P.,2/15/2013,WD
4,Fitch Ratings,ESB Finance Designated Activity Company,11/13/2012,BBB+
...,...,...,...,...
474380,Standard & Poor's Ratings Services,Tops Markets II Corporation,12/13/2016,B-
474417,DBRS,Daimler Canada Finance Inc.,1/22/1996,R-1 (middle)
474507,Standard & Poor's Ratings Services,Essilor International S.A.,3/7/2014,A-1
474579,Fitch Ratings,Wales & West Utilities Limited,6/15/2012,A-


In [11]:
df_is_stocks

,Symbol,Name,Sector,industry
0,A,"Agilent Technologies, Inc.",Capital Goods,Biotechnology: Laboratory Analytical Instruments
1,AA,Alcoa Corporation,Basic Industries,Aluminum
2,AACG,ATA Creativity Global,Consumer Services,Other Consumer Services
3,AAL,"American Airlines Group, Inc.",Transportation,Air Freight/Delivery Services
4,AAMC,Altisource Asset Management Corp,Finance,Real Estate
...,...,...,...,...
7017,ZVO,Zovio Inc.,Consumer Services,Other Consumer Services
7018,ZYME,Zymeworks Inc.,Health Care,Major Pharmaceuticals
7019,ZYNE,"Zynerba Pharmaceuticals, Inc.",Health Care,Major Pharmaceuticals
7020,ZYXI,"Zynex, Inc.",Health Care,Biotechnology: Electromedical & Electrotherape...


In [4]:

# Apply transformation
df_ratings['Rating'] = df_ratings['Rating'].apply(str).apply(standarizeRating)

# We do not wan't to consider if a Rating was A+ or A. 
# We remove  + and - from ratings
df_ratings['Rating'] = df_ratings['Rating'].str.replace(r'+', '').str.replace(r'-', '')

df_ratings

,Rating Agency Name,Name,Rating Action Date,Rating
0,Egan-Jones Ratings Company,TELECOM ITALIA S.P.A,8/26/2014,BB
1,Egan-Jones Ratings Company,WHIRLPOOL CORPORATION,11/27/2015,A
2,Egan-Jones Ratings Company,UNITED TECHNOLOGIES,7/28/2014,A
3,Fitch Ratings,TEPPCO Partners L.P.,2/15/2013,WD
4,Fitch Ratings,ESB Finance Designated Activity Company,11/13/2012,BBB
...,...,...,...,...
474380,Standard & Poor's Ratings Services,Tops Markets II Corporation,12/13/2016,B
474417,DBRS,Daimler Canada Finance Inc.,1/22/1996,R1 (middle)
474507,Standard & Poor's Ratings Services,Essilor International S.A.,3/7/2014,A1
474579,Fitch Ratings,Wales & West Utilities Limited,6/15/2012,A


In [6]:
# Filter Ratings
# Exclude non-traditional ratings from the datasets
rating_set = {'A', 'AA', 'AAA', 
              'B', 'BB', 'BBB', 
              'C', 'CC', 'CCC', 
              'D'}

df_ratings = df_ratings[df_ratings['Rating'].isin(rating_set)]

# Filter duplicates in the dataset
df_ratings = df_ratings.drop_duplicates(subset = df_ratings.columns)

df_ratings.groupby('Rating').count()

df_ratings

,Rating Agency Name,Name,Rating Action Date,Rating
2,Egan-Jones Ratings Company,UNITED TECHNOLOGIES,7/28/2014,A
11,Egan-Jones Ratings Company,SCHLUMBERGER NV,7/18/2014,AA
18,Egan-Jones Ratings Company,SYSCO CORPORATION,6/1/2015,A
21,Egan-Jones Ratings Company,COMCAST CORPORATION CLASS A,6/3/2015,A
22,Standard & Poor's Ratings Services,Presidio Inc.,2/2/2015,B
...,...,...,...,...
472649,Fitch Ratings,United Airlines Pass Through Trust Series 2014-2,10/25/2016,BBB
472742,Fitch Ratings,US Airways 2012-1 Pass Through Trust,12/7/2015,A
473277,DBRS,Canadian Oil Sands Limited,8/3/2005,BBB
473284,Fitch Ratings,Global Switch Property (Australia) Pty Limited,1/13/2014,BBB


In [13]:
# Clean company names
# Remove unwated characters from the names
# We want to merge the datasets df_is_stocks and df_ratings
# The best method is to remove all possible characters that will cause mismatch
# This function will remove all characters and names we don't want


def clean(name):
    
    name = name.lower() # Convert to lowercase
    name = re.sub(r'\.', '', name) # Remove periods
    name = re.sub(r',', '', name) # Remove commas
    name = re.sub(r'^a-z', '', name) # Remove leading letters
    name = re.sub(r'corporation', '', name)
    name = re.sub(r' corp', '', name)    
    name = re.sub(r' co', '', name)   
    name = re.sub(r'inc', '', name)
    name = re.sub(r'limited', '', name)
    name = re.sub(r'ltd', '', name)
    name = re.sub(r'holdings', '', name)
    name = re.sub(r' holding', '', name)    
    name = re.sub(r'plc', '', name)
    name = re.sub(r'group', '', name)
    name = re.sub(r' ag', '', name)
    name = re.sub(r' sa', '', name)
    name = re.sub(r' pty', '', name)
    name = re.sub(r' international', '', name)
    name = re.sub(r' incorporated', '', name)
    name = re.sub(r' spa', '', name)
    name = re.sub(r' se', '', name)
    name = re.sub(r' lp', '', name)    
    name = re.sub(r' (The)', '', name)  # Remove "The" at the beginning of the name  
    name = re.sub(r'The', '', name)    
    name = re.sub(r'LLC', '', name)   
    name = re.sub(r'n.v', '', name) 
    name = name.strip() # Remove leading and trailing whitespace
    
    return name


# Apply the function in both datasets
df_is_stocks.loc[:, 'Clean_Name'] = df_is_stocks['Name']
df_is_stocks.loc[:, 'Clean_Name'] = df_is_stocks['Name'].apply(clean)
df_ratings.loc[:, 'Clean_Name'] = df_ratings['Name']
df_ratings.loc[:, 'Clean_Name'] = df_ratings['Clean_Name'].apply(str).apply(clean)

df_is_stocks

,Symbol,Name,Sector,industry,Clean_Name
0,A,"Agilent Technologies, Inc.",Capital Goods,Biotechnology: Laboratory Analytical Instruments,agilent technologies
1,AA,Alcoa Corporation,Basic Industries,Aluminum,alcoa
2,AACG,ATA Creativity Global,Consumer Services,Other Consumer Services,ata creativity global
3,AAL,"American Airlines Group, Inc.",Transportation,Air Freight/Delivery Services,american airlines
4,AAMC,Altisource Asset Management Corp,Finance,Real Estate,altisource asset management
...,...,...,...,...,...
7017,ZVO,Zovio Inc.,Consumer Services,Other Consumer Services,zovio
7018,ZYME,Zymeworks Inc.,Health Care,Major Pharmaceuticals,zymeworks
7019,ZYNE,"Zynerba Pharmaceuticals, Inc.",Health Care,Major Pharmaceuticals,zynerba pharmaceuticals
7020,ZYXI,"Zynex, Inc.",Health Care,Biotechnology: Electromedical & Electrotherape...,zynex


In [14]:
# Merge Datasets
df_ratings = pd.merge(df_ratings, df_is_stocks ,on='Clean_Name')


# Clean the table from the merge
df_ratings = df_ratings.rename(columns = {'Name_x': 'Name'}).drop('Name_y', axis = 1)

df_ratings

,Rating Agency Name,Name,Rating Action Date,Rating,Clean_Name,Symbol,Sector,industry
0,Egan-Jones Ratings Company,WHIRLPOOL CORPORATION,11/27/2015,A-,whirlpool,WHR,Consumer Durables,Consumer Electronics/Appliances
1,Fitch Ratings,Kellogg Company,6/15/2012,F2,kelloggmpany,K,Consumer Non-Durables,Packaged Foods
2,Egan-Jones Ratings Company,SCHLUMBERGER NV,7/18/2014,AA,schlumberger nv,SLB,Energy,Oilfield Services/Equipment
3,Egan-Jones Ratings Company,HONEYWELL INTERNATIONAL INC,4/29/2014,AA-,honeywell,HON,Capital Goods,Aerospace
4,Egan-Jones Ratings Company,SYSCO CORPORATION,6/1/2015,A,sysco,SYY,Consumer Non-Durables,Food Distributors
...,...,...,...,...,...,...,...,...
19584,Fitch Ratings,Vulcan Materials Company,12/15/2015,NR,vulcan materialsmpany,VMC,Basic Industries,Mining & Quarrying of Nonmetallic Minerals (No...
19585,Standard & Poor's Ratings Services,Gerdau Holdings Inc.,8/2/2013,BBB-,gerdau,GGB,Capital Goods,Steel/Iron Ore
19586,Fitch Ratings,ACCO Brands Corporation,4/3/2013,BB,acco brands,ACCO,Consumer Durables,Publishing
19587,Fitch Ratings,DXC Technology Company,12/28/2012,BBB,dxc technologympany,DXC,Technology,EDP Services


In [15]:
# Visualise the data
df_ratings.groupby('Rating').count()

df_ratings = df_ratings.drop_duplicates(subset = ['Rating Action Date', 'Clean_Name'])

df_ratings

,Rating Agency Name,Name,Rating Action Date,Rating,Clean_Name,Symbol,Sector,industry
0,Egan-Jones Ratings Company,WHIRLPOOL CORPORATION,11/27/2015,A-,whirlpool,WHR,Consumer Durables,Consumer Electronics/Appliances
1,Fitch Ratings,Kellogg Company,6/15/2012,F2,kelloggmpany,K,Consumer Non-Durables,Packaged Foods
2,Egan-Jones Ratings Company,SCHLUMBERGER NV,7/18/2014,AA,schlumberger nv,SLB,Energy,Oilfield Services/Equipment
3,Egan-Jones Ratings Company,HONEYWELL INTERNATIONAL INC,4/29/2014,AA-,honeywell,HON,Capital Goods,Aerospace
4,Egan-Jones Ratings Company,SYSCO CORPORATION,6/1/2015,A,sysco,SYY,Consumer Non-Durables,Food Distributors
...,...,...,...,...,...,...,...,...
19575,Fitch Ratings,Thermo Fisher Scientific Inc.,3/1/2016,WD,thermo fisher scientific,TMO,Capital Goods,Industrial Machinery/Components
19585,Standard & Poor's Ratings Services,Gerdau Holdings Inc.,8/2/2013,BBB-,gerdau,GGB,Capital Goods,Steel/Iron Ore
19586,Fitch Ratings,ACCO Brands Corporation,4/3/2013,BB,acco brands,ACCO,Consumer Durables,Publishing
19587,Fitch Ratings,DXC Technology Company,12/28/2012,BBB,dxc technologympany,DXC,Technology,EDP Services


In [ ]:
####################################################
####################################################
####################################################
####################################################
# Now we will get the financial
# We will use the API Financial Modeling Prep
# Create function to get data

def get_jsonparsed_data(url):
    
    response = urlopen(url)
    data = response.read().decode("utf-8")
    return json.loads(data)


# List of the symbols/companies 
Symbol_list = list(set(df_ratings.Symbol))

# Remove Unwated characters
Symbol_list = [re.sub(r'\^.+', '', item) for item in Symbol_list]

# Create a dataframe to place the the data
df_company_ratios = pd.DataFrame.from_dict(
        get_jsonparsed_data("https://financialmodelingprep.com/api/v3/ratios/" 
                            + Symbol_list[0] + 
                            "?apikey=c283e7eb3ddc10dcc0ed8146046dff97"))

df_company_ratios = df_company_ratios[0:0] # empty the dataframe

# Get every company from the list the financial information
for Symbol in Symbol_list:
    
    if Symbol.isalpha() == True:
    
        ratios = pd.DataFrame.from_dict(
                get_jsonparsed_data("https://financialmodelingprep.com/api/v3/ratios/" + 
                                    Symbol + 
                                    "?apikey=c283e7eb3ddc10dcc0ed8146046dff97"))
    
        frames = [df_company_ratios, ratios]
    
        df_company_ratios = pd.concat(frames)   



# Now filter by date
# The api returns every available information from the past
# we want exclusively the data from the year of the rating or the year before
# If the rating was done before june we will take the indicators from the year before
# If it was after june we will take from the following year
df_ratings = df_ratings.rename(columns = {'Rating Action Date':'date'})
df_ratings['date'] = df_ratings['date'].astype('datetime64[ns]')
df_ratings['month_rating'] = df_ratings['date'].dt.month
df_ratings['year_change'] = [1 if x < 7 else 0 for x in df_ratings['month_rating']]
df_ratings['year'] = df_ratings['date'].dt.year
df_ratings['Year'] = df_ratings['year'] - df_ratings['year_change']

df_company_ratios = df_company_ratios.rename(columns = {'symbol':'Symbol'})
df_company_ratios['date'] = df_company_ratios['date'].astype('datetime64[ns]')
df_company_ratios['Year'] = df_company_ratios['date'].dt.year


# merge Datasets
df_ratings = pd.merge(df_ratings, df_company_ratios, on=['Symbol', 'Year']) 

# Work on missing values and eliminate duplicates
a = df_ratings.count()
the_list = a[a<5900].reset_index()['index']
df_ratings_NA = df_ratings[df_ratings.columns.difference(the_list)]
df_ratings_NA = df_ratings_NA.dropna()
df_ratings_NA = df_ratings_NA.drop_duplicates(['Clean_Name', 'date_y'])


# clean the dataset - drop columns
df_ratings_NA = df_ratings_NA.drop(['Name', 'Year', 'date_y', 'ebtPerEbit',
                                    'month_rating', 'netIncomePerEBT', 'year',
                                    'year_change', 'industry'
                                    'priceCashFlowRatio', 'priceEarningsRatio',	
                                    'priceSalesRatio',
                                    'priceToFreeCashFlowsRatio',	
                                    'priceToOperatingCashFlowsRatio',	
                                    'priceToSalesRatio'], axis=1)


# clean the dataset - Rename columns
df_ratings_NA = df_ratings_NA.rename({'Clean_Name' :'Name', 'date_x':'Date'})

# Export to CSVDatasets
df_ratings_NA.to_csv (r'data/rating_dataframe.csv', index = False, header=True)






